In [10]:
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
data=pd.read_csv("/content/train.csv")
df=pd.DataFrame(data)
print(df.isnull().sum())
print(df.head())
print(df["meal_id"].nunique())
print(df["center_id"].nunique())

id                       0
week                     0
center_id                0
meal_id                  0
checkout_price           0
base_price               0
emailer_for_promotion    0
homepage_featured        0
num_orders               0
dtype: int64
        id  week  center_id  meal_id  checkout_price  base_price  \
0  1379560     1         55     1885          136.83      152.29   
1  1466964     1         55     1993          136.83      135.83   
2  1346989     1         55     2539          134.86      135.86   
3  1338232     1         55     2139          339.50      437.53   
4  1448490     1         55     2631          243.50      242.50   

   emailer_for_promotion  homepage_featured  num_orders  
0                      0                  0         177  
1                      0                  0         270  
2                      0                  0         189  
3                      0                  0          54  
4                      0                  0  

In [12]:
df["discount"] = (df["base_price"] - df["checkout_price"])/df["base_price"]
df = pd.get_dummies(df, columns=["meal_id","center_id"])
df["promo_discount"] =df["discount"] * df["homepage_featured"]*df["emailer_for_promotion"]
X=df.drop(["num_orders","id"],axis=1)
Y=df["num_orders"]
print(X.shape)

(456548, 135)


In [15]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)
clf=XGBRegressor(random_state=42,eval_metric="rmse")
clf.fit(X_train,Y_train)
print(clf.score(X_train,Y_train))
print(clf.score(X_test,Y_test))

Y_pred=clf.predict(X_test)
print(mean_squared_error(Y_test,Y_pred))
print(r2_score(Y_train, clf.predict(X_train)))
print(r2_score(Y_test, Y_pred))

0.8417306542396545
0.802120566368103
30183.23046875
0.8417306542396545
0.802120566368103


In [14]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": clf.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

print(importance.head(20))

                   feature  importance
36            meal_id_2290    0.048767
22            meal_id_1754    0.036752
31            meal_id_1971    0.032768
128          center_id_157    0.032566
28            meal_id_1885    0.029514
13            meal_id_1248    0.029005
4        homepage_featured    0.027651
32            meal_id_1993    0.025023
35            meal_id_2139    0.024474
38            meal_id_2306    0.021852
48            meal_id_2631    0.021365
75            center_id_43    0.020724
3    emailer_for_promotion    0.020594
50            meal_id_2664    0.020445
21            meal_id_1727    0.019394
134         promo_discount    0.018654
59            center_id_13    0.017856
120          center_id_137    0.017194
131          center_id_174    0.016742
7             meal_id_1109    0.014547
